In [0]:
# Notebook 03 - Gold
# Create Star Schema from Silver layer data

SILVER_PATH = "/Volumes/insurance_project/silver/clean_data"

df_insurance = spark.read.parquet(f"{SILVER_PATH}/insurance_data")
df_employees = spark.read.parquet(f"{SILVER_PATH}/employee_data")
df_vendors = spark.read.parquet(f"{SILVER_PATH}/vendor_data")

print("Silver data loaded!")

Silver data loaded!


In [0]:
# Create dimension tables

# dim_customers - who bought the policy

dim_customers = df_insurance.select(
    "CUSTOMER_ID", "CUSTOMER_NAME", "AGE", "MARITAL_STATUS",
    "EMPLOYMENT_STATUS", "NO_OF_FAMILY_MEMBERS", "SOCIAL_CLASS",
    "CUSTOMER_EDUCATION_LEVEL", "HOUSE_TYPE", "CITY", "STATE"
).dropDuplicates(["CUSTOMER_ID"])

# dim_policies - what type of policy

dim_policies = df_insurance.select(
    "POLICY_NUMBER", "INSURANCE_TYPE", "PREMIUM_AMOUNT", "TENURE"
).dropDuplicates(["POLICY_NUMBER"])

# dim_agents - who handled the claim

dim_agents = df_employees.select(
    "AGENT_ID", "AGENT_NAME", "CITY", "STATE"
).dropDuplicates(["AGENT_ID"])

# dim_vendors - external experts involved

dim_vendors = df_vendors.select(
    "VENDOR_ID", "VENDOR_NAME", "CITY", "STATE"
).dropDuplicates(["VENDOR_ID"])

print("Dimension tables created!")
print("dim_customers:", dim_customers.count(), "rows")
print("dim_policies:", dim_policies.count(), "rows")
print("dim_agents:", dim_agents.count(), "rows")
print("dim_vendors:", dim_vendors.count(), "rows")

Dimension tables created!
dim_customers: 10000 rows
dim_policies: 10000 rows
dim_agents: 1200 rows
dim_vendors: 600 rows


In [0]:
# Create dim_dates from transaction dates

from pyspark.sql.functions import col, year, month, dayofmonth, dayofweek, date_format

dim_dates = df_insurance.select("TXN_DATE_TIME").dropDuplicates().select(
    col("TXN_DATE_TIME").alias("date"),
    year("TXN_DATE_TIME").alias("year"),
    month("TXN_DATE_TIME").alias("month"),
    dayofmonth("TXN_DATE_TIME").alias("day"),
    dayofweek("TXN_DATE_TIME").alias("day_of_week"),
    date_format("TXN_DATE_TIME", "EEEE").alias("day_name"),
    date_format("TXN_DATE_TIME", "MMMM").alias("month_name")
)

print("dim_dates created:", dim_dates.count(), "rows")

dim_dates created: 395 rows


In [0]:
# Create fact_claims - central fact table

fact_claims = df_insurance.select(
    "TRANSACTION_ID",
    "CUSTOMER_ID",
    "POLICY_NUMBER",
    "AGENT_ID",
    "VENDOR_ID",
    "TXN_DATE_TIME",
    "LOSS_DT",
    "REPORT_DT",
    "CLAIM_AMOUNT",
    "CLAIM_STATUS",
    "INCIDENT_SEVERITY",
    "AUTHORITY_CONTACTED",
    "ANY_INJURY",
    "POLICE_REPORT_AVAILABLE",
    "INCIDENT_STATE",
    "INCIDENT_CITY",
    "INCIDENT_HOUR_OF_THE_DAY"
)

print("fact_claims created:", fact_claims.count(), "rows")

fact_claims created: 10000 rows


In [0]:
# Add fraud indicators to fact_claims

from pyspark.sql.functions import avg, when

# Calculate average claim amount

avg_claim = df_insurance.agg(avg("CLAIM_AMOUNT")).collect()[0][0]

# Apply fraud rules

fact_claims = fact_claims.withColumn(
    "fraud_indicator",
    when(
        (col("CLAIM_AMOUNT") > avg_claim * 2) &
        (col("POLICE_REPORT_AVAILABLE") == 0), 1
    ).when(
        (col("INCIDENT_SEVERITY") == "Total Loss") &
        (col("ANY_INJURY") == 0), 1
    ).otherwise(0)
)

# Apply risk level based on claim amount

fact_claims = fact_claims.withColumn(
    "risk_level",
    when(col("CLAIM_AMOUNT") > avg_claim * 2, "High")
    .when(col("CLAIM_AMOUNT") > avg_claim, "Medium")
    .otherwise("Low")
)

print("Fraud indicators added!")
print("Average claim amount:", round(avg_claim, 2))

Fraud indicators added!
Average claim amount: 16563.83


In [0]:
# Save all Gold tables to Parquet

GOLD_PATH = "/Volumes/insurance_project/gold/star_schema"

dim_customers.write.mode("overwrite").parquet(f"{GOLD_PATH}/dim_customers")
dim_policies.write.mode("overwrite").parquet(f"{GOLD_PATH}/dim_policies")
dim_agents.write.mode("overwrite").parquet(f"{GOLD_PATH}/dim_agents")
dim_vendors.write.mode("overwrite").parquet(f"{GOLD_PATH}/dim_vendors")
dim_dates.write.mode("overwrite").parquet(f"{GOLD_PATH}/dim_dates")
fact_claims.write.mode("overwrite").parquet(f"{GOLD_PATH}/fact_claims")

print("Gold layer saved successfully!")

Gold layer saved successfully!
